In [1]:
# 토크나이즈 허깅페이스에서 다운로드하기
from transformers import AutoTokenizer

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [2]:
# 감정분석 => 영화리뷰를 읽어서 긍정/부정인지 판별
import pandas as pd

train = pd.read_csv("http://114.207.245.181:13000/txt/ratings_train.txt", sep="\t")
test = pd.read_csv("http://114.207.245.181:13000/txt/ratings_test.txt", sep="\t")

In [3]:
# 결측치 제거
train = train.dropna()
test = test.dropna()

# 문장과 라벨 분리
x_train = train['document'].to_list()
y_train = train['label'].values

x_test = test['document'].to_list()
y_test = test['label'].values

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 텍스트 토큰화
tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(x_train)
# 정수 인코딩
x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

In [18]:
max_len = 50

x_train_pad = pad_sequences(
    x_train_seq,
    maxlen=max_len,
    padding='post'
)

x_test_pad = pad_sequences(
    x_test_seq,
    maxlen=max_len,
    padding='post'
)

vocab_size = len(tokenizer.word_index) + 1

vocab_size

296312

In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(100,)),
    Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True),
    LSTM(units=64, activation="tanh", dropout=0.3, recurrent_dropout=0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 100, 128)       │    37,927,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,979,457 (144.88 MB)

 Trainable params: 37,979,457 (144.88 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(x_train_pad, y_train, validation_data=(x_test_pad, y_test), epochs=5, batch_size=64)

Epoch 1/5
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 422s 179ms/step - accuracy: 0.8039 - loss: 0.4084 - val_accuracy: 0.8272 - val_loss: 0.3653
Epoch 2/5
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 487s 208ms/step - accuracy: 0.9443 - loss: 0.1514 - val_accuracy: 0.8151 - val_loss: 0.4393
Epoch 3/5
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 481s 205ms/step - accuracy: 0.9765 - loss: 0.0675 - val_accuracy: 0.8160 - val_loss: 0.5220
Epoch 4/5
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 499s 213ms/step - accuracy: 0.9866 - loss: 0.0380 - val_accuracy: 0.8119 - val_loss: 0.6259
Epoch 5/5
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 486s 207ms/step - accuracy: 0.9906 - loss: 0.0257 - val_accuracy: 0.8108 - val_loss: 0.7213


In [39]:
model.evaluate(x_test_pad, y_test)

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8108 - loss: 0.7213


[0.721257746219635, 0.8107686638832092]

In [48]:
pred_texts = [
    "이 영화 너무너무 좋아요",
    "배우 연기력 짱짱맨이네요",
    "토나오는 영화"
]

pred_seq = tokenizer.texts_to_sequences(pred_texts)

pred_pad = pad_sequences(pred_seq, maxlen=max_len, padding='post')

pred = model.predict(pred_pad)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


In [49]:
for text, p in zip(pred_texts, pred):
    if p[0] > 0.5:
        sentiment = "긍정"
    else:
        sentiment = "부정"

    print(f"{text} -> {sentiment} ({p[0]:.4f})")

이 영화 너무너무 좋아요 -> 긍정 (0.9870)
배우 연기력 짱짱맨이네요 -> 부정 (0.3674)
토나오는 영화 -> 부정 (0.0287)
